# 09 — Independent day-3 transient MCAO spatial screen (GSE337578)

**Source**: GEO GSE337578, GSM9858163, mouse 60-minute transient MCAO, 3 days post-reperfusion; one deposited spatial library on platform GPL35959 (released 2026).

**Purpose**: add a second ischemia-model spatial dataset with barcode coordinates to the IMACS target-adjudication evidence ledger.

**Strict interpretation**: no sham/control library, no replicated animals, no pathological core/penumbra mask validated here. **No differential-expression p-values, no biological variance, and no lesion-core localization claims.** Spatially varying transcript detection is descriptive, may reflect cell mixture, tissue morphology, RNA capture, and missing cells. <https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE337578>


In [ ]:
!pip -q install "numpy>=1.26" "pandas>=2.1" "scipy>=1.12" "matplotlib>=3.8" requests

import os,gzip,hashlib,json,sys
from pathlib import Path
import numpy as np, pandas as pd,scipy.sparse as sp,requests
from scipy.io import mmread
import matplotlib.pyplot as plt

ROOT=Path(os.environ.get("IMACS_ROOT","/content/IMACS_GSE337578" if Path("/content").exists() else "./IMACS_GSE337578"))
RAW=ROOT/"raw"; OUT=ROOT/"results"; FIG=ROOT/"figures"
for x in (RAW,OUT,FIG):x.mkdir(parents=True,exist_ok=True)
gsm="GSM9858163"
base="https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM9858nnn/GSM9858163/suppl/"
prefix=gsm+"_"
files=["matrix.mtx.gz","barcodes.tsv.gz","features.tsv.gz","cell_locations.tsv.gz"]
def dl(filename):
    p=RAW/(prefix+filename)
    if not p.is_file() or p.stat().st_size==0:
        with requests.get(base+prefix+filename,stream=True,timeout=300) as r:
            r.raise_for_status()
            with p.open("wb") as f:
                for part in r.iter_content(chunk_size=1<<20):
                    if part:f.write(part)
    return p
ps={x:dl(x) for x in files}
def sha256(p):
 h=hashlib.sha256()
 with open(p,"rb") as f:
  for chunk in iter(lambda:f.read(1<<20),b""):h.update(chunk)
 return h.hexdigest()
pd.DataFrame([{"accession":gsm,"filename":p.name,"bytes":p.stat().st_size,"sha256":sha256(p)} for p in ps.values()]).to_csv(OUT/"09_spatial_input_checksums.csv",index=False)
print([(x,ps[x].stat().st_size) for x in files])


In [ ]:
with gzip.open(ps["matrix.mtx.gz"],"rb") as f: X=mmread(f)
with gzip.open(ps["barcodes.tsv.gz"],"rt") as f: bcs=[line.strip() for line in f if line.strip()]
features=pd.read_csv(ps["features.tsv.gz"],compression="gzip",header=None,sep="\t")
genes=features.iloc[:,1].astype(str).values if features.shape[1]>1 else features.iloc[:,0].astype(str).values
if X.shape==(len(genes),len(bcs)):
 counts=X.tocsr()
elif X.shape==(len(bcs),len(genes)):
 counts=X.T.tocsr()
else: raise AssertionError(f"Unexpected matrix {X.shape}, genes={len(genes)}, barcodes={len(bcs)}")
assert counts.shape==(len(genes),len(bcs))
print("genes,barcodes,nonzeros",counts.shape,counts.nnz)
CATALOG_URL="https://raw.githubusercontent.com/Touf00/retinal-nvu-computational-analysis/imacs-stroke-pet-screen/IMACS_Stroke_PET/config/candidate_catalog.csv"
cat=pd.read_csv(CATALOG_URL).drop_duplicates("gene")
print("frozen candidates:",len(cat))


## Spatial-coordinate format audit

The uploaded cell-location table defines spatial coordinates. Do not guess X/Y or swap them silently. The code saves the original header and flags any ambiguity. If the coordinate columns are not unambiguously identifiable, output candidate counts but halt plotting, with the file ready for manual review.

In [ ]:
loc=pd.read_csv(ps["cell_locations.tsv.gz"],compression="gzip",sep="\t")
print("Coordinates table columns:",list(loc.columns))
display(loc.head())
(OUT/"09_coordinates_column_schema.json").write_text(json.dumps({"columns":list(loc.columns),"n_rows":int(len(loc))},indent=2))
possible_barcode=[c for c in loc.columns if c.lower() in ("barcode","barcodes","cell_id","cellid","cell","spot_id")]
if not possible_barcode: possible_barcode=[c for c in loc.columns if "barcode" in c.lower()]
if not possible_barcode: raise ValueError("No unambiguous cell barcode column: inspect saved schema, do not fake spatial matches")
bc_col=possible_barcode[0]
loc[bc_col]=loc[bc_col].astype(str)
u=set(bcs)
nmatched=loc[bc_col].isin(u).sum()
assert nmatched>=.5*min(len(loc),len(bcs)),f"Coordinate barcode overlap is low: {nmatched}/{len(loc)}"
loc=loc.drop_duplicates(bc_col).set_index(bc_col).reindex(bcs)
coordcols=list(loc.columns)
x_candidates=[c for c in coordcols if c.lower() in ("x","pos_x","x_coordinate","xcoord","spatial_x","pixel_x","pxl_col_in_fullres")]
y_candidates=[c for c in coordcols if c.lower() in ("y","pos_y","y_coordinate","ycoord","spatial_y","pixel_y","pxl_row_in_fullres")]
if len(x_candidates)!=1 or len(y_candidates)!=1:
 print("COORDINATE FORMAT UNRESOLVED; no spatial plot. Columns:",coordcols)
 xcol=ycol=None
else:
 xcol=x_candidates[0];ycol=y_candidates[0]
 print("Coordinate columns",xcol,ycol)


## Prespecified target-panel expression

Normalize each spatial unit by its whole-transcript count and report raw detection, mean log-normalized counts and spatial heterogeneity. This is **not single-cell RNA-seq**; the number of nuclei/transcripts per unit varies.

In [ ]:
library=np.asarray(counts.sum(axis=0)).ravel().astype(float)
rows=[]; point_arrays={}
features_idx=pd.Series(np.arange(len(genes)),index=genes)
for g in cat["gene"]:
 ids=np.flatnonzero(genes==g)
 if len(ids)==0:
  rows.append({"gene":g,"matching_features":0,"n_spatial_units":len(bcs),"pct_detected":np.nan,"mean_log1pCP10K":np.nan,"status":"FEATURE_NOT_MATCHED"})
  continue
 y=np.asarray(counts[ids,:].sum(axis=0)).ravel().astype(float)
 signal=np.log1p(np.divide(y,library,out=np.zeros_like(y),where=library>0)*10000.0)
 rows.append({"gene":g,"matching_features":len(ids),"n_spatial_units":len(bcs),"pct_detected":float((y>0).mean()*100),
             "mean_log1pCP10K":float(signal.mean()),"status":"DETECTED_OR_TESTABLE"})
 point_arrays[g]=signal
evidence=pd.DataFrame(rows)
evidence=evidence.merge(cat,on="gene",how="left",validate="one_to_one")
evidence.to_csv(OUT/"09_day3_spatial_frozen_panel_descriptive.csv",index=False)
pd.DataFrame({"barcode":bcs,"whole_spatial_unit_UMI":library}).to_csv(OUT/"09_spatial_capture_depth_by_barcode.csv",index=False)
display(evidence)


In [ ]:
if xcol and ycol:
  x=pd.to_numeric(loc[xcol],errors="coerce").to_numpy()
  y=pd.to_numeric(loc[ycol],errors="coerce").to_numpy()
  good=np.isfinite(x)&np.isfinite(y)&(library>0)
  assert good.sum()>.5*len(bcs),"Too few usable coordinates"
  print("Spatial units with coordinates:",good.sum())
  pd.DataFrame({"barcode":bcs,"x":x,"y":y,"library_UMI":library}).to_csv(OUT/"09_spatial_coordinates_audited.csv",index=False)
  for target in ["Aplnr","Amot","Mcam","Maob","Mfsd2a","Tfrc","Slc3a2","Ackr1","Icam1"]:
   if target not in point_arrays:continue
   fig,ax=plt.subplots(figsize=(7,7))
   q=ax.scatter(x[good],y[good],c=point_arrays[target][good],s=2,cmap="viridis",rasterized=True)
   ax.set_title(f"{target}: normalized RNA, day-3 tMCAO (one sample)")
   ax.set_aspect("equal");ax.invert_yaxis()
   ax.set_xlabel(xcol);ax.set_ylabel(ycol)
   fig.colorbar(q,label="log1p counts/10k")
   fig.savefig(FIG/f"09_day3_{target}_spatial_descriptive.png",dpi=140,bbox_inches="tight")
   plt.close(fig)
else:
 (OUT/"09_NO_SPATIAL_MAP_UNVALIDATED_COORDINATES.txt").write_text(
 "Coordinates could not be determined unambiguously from GEO header; no maps generated.")
print("Validated spatial schema completed")


In [ ]:
manifest={"dataset":"GSE337578","sample":"GSM9858163","model":"mouse 60-minute tMCAO day3",
"replicate_unit":"ONE SPATIAL LIBRARY; no biological inferential statistics",
"source_accession":"https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE337578",
"n_barcodes":len(bcs),"n_features":len(genes),
"coordinate_valid":bool(xcol and ycol),
"RNA_only_no_core_mask":True}
(OUT/"09_spatial_run_manifest.json").write_text(json.dumps(manifest,indent=2))
print(manifest)
